In [ ]:
# ============================================================
# CELL 3 (TUY CHON): Chuan hoa am luong EBU R128
# ============================================================
# Chay cell nay SAU KHI pipeline hoan thanh (Cell 2)
# EBU R128: Integrated = -16 LUFS, True Peak = -1.5 dBTP

import os
from google.colab import files

input_mp3 = "/tmp/srt_output/output.mp3"
output_mp3 = "/tmp/srt_output/output_normalized.mp3"

if not os.path.exists(input_mp3):
    print(f'❌ Chua co file output.mp3 — chay Cell 2 truoc da!')
else:
    !ffmpeg -i {input_mp3} \
            -af loudnorm=I=-16:TP=-1.5:LRA=11:print_format=summary \
            {output_mp3} -y 2>&1 | grep -E "Input|Output"

    if os.path.exists(output_mp3):
        size_mb = os.path.getsize(output_mp3) / (1024 * 1024)
        print(f'✅ Da chuan hoa! File: {size_mb:.1f}MB')
        files.download(output_mp3)
    else:
        print('❌ Chuan hoa that bai — thu lai Cell 2 truoc.')

In [ ]:
# ============================================================
# CELL 2: Khoi dong Model + Giao dien + Pipeline
# ============================================================
import logging, os, re, time, tempfile, subprocess, wave
import numpy as np
import torch
import IPython
from IPython.display import HTML, display, Javascript
from google.colab import output, files

# ─── Shims ───────────────────────────────────────────
import torch as _torch
if not hasattr(_torch, '_utils'):
    _torch._utils = _torch._C._utils

import transformers as _tf
class _SafeAutoFeatureExtractor:
    @staticmethod
    def from_pretrained(model_name, **kwargs):
        try:
            from transformers import AutoConfig
            cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True, **kwargs)
            sr = getattr(cfg, 'sampling_rate', 24000)
        except Exception:
            sr = 24000
        class _Result:
            sampling_rate = sr
        return _Result()
_tf.AutoFeatureExtractor = _SafeAutoFeatureExtractor

from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.common import get_best_device

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# ─── Load OmniVoice Model ─────────────────────────────
print('🔍 Dang kiem tra GPU...')
for i in range(30):
    if torch.cuda.is_available():
        print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
        break
    time.sleep(1)
else:
    print('⚠️ GPU khong kha dung — vao Runtime > Change runtime type > T4 GPU')

DEVICE = get_best_device()
print(f'🚀 Dang tai OmniVoice tren {DEVICE}... (lan dau ~5 phut)')
model = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice', device_map=DEVICE, dtype=torch.float16, load_asr=True
)
SAMPLING_RATE = model.sampling_rate
print(f'✅ Model san sang — Sample rate: {SAMPLING_RATE}Hz')

# ─── Tao 7 Voice Clone Prompts ────────────────────────
print('\n🎤 Dang tao voice clone prompts...')
VOICE_PROMPTS = {}
for key in VOICE_KEYS:
    sample_path = f"/tmp/srt_voices/{key}.mp3"
    if os.path.exists(sample_path):
        print(f'  ⏳ {VOICE_NAMES[key]}...')
        VOICE_PROMPTS[key] = model.create_voice_clone_prompt(ref_audio=sample_path)
        print(f'  ✅ {VOICE_NAMES[key]}')
    else:
        print(f'  ⚠️ {VOICE_NAMES[key]} — KHONG TIM THAY sample!')
print(f'✅ {len(VOICE_PROMPTS)}/7 voice prompts san sang!')

# ─── Generation Config ────────────────────────────────
GEN_CFG = OmniVoiceGenerationConfig(
    num_step=32, guidance_scale=1.8,
    denoise=True, preprocess_prompt=True, postprocess_output=True,
    position_temperature=5.0, class_temperature=0.2,
    pad_duration=0.1, fade_duration=0.1,
)

# ─── Parse SRT ────────────────────────────────────────
def parse_srt(text):
    """Parse SRT text -> list[{index, start_s, end_s, text}]"""
    if text.startswith('﻿'):
        text = text[1:]
    blocks = re.split(r'\n\s*\n', text.strip())
    lines = []
    def _ts(s):
        s = s.replace(',', '.')
        h, m, sec = s.split(':')
        return int(h)*3600 + int(m)*60 + float(sec)
    for block in blocks:
        m = re.match(
            r'(\d+)\s*\n\s*(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*'
            r'(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*\n\s*(.+)',
            block.strip(), re.DOTALL
        )
        if m:
            txt = re.sub(r'<[^>]+>', '', m.group(4).strip())
            lines.append({
                'index': int(m.group(1)),
                'start_s': _ts(m.group(2)),
                'end_s':   _ts(m.group(3)),
                'text':    txt.replace('\n', ' ')
            })
    return lines

# ─── WAV Writer (built-in, no deps) ───────────────────
def write_wav(path, audio, sr):
    """Save numpy float audio [-1,1] as 16-bit mono WAV"""
    wf = (audio * 32767).astype(np.int16)
    with wave.open(path, 'w') as f:
        f.setnchannels(1); f.setsampwidth(2); f.setframerate(sr)
        f.writeframes(wf.tobytes())

# ─── JS Helpers ───────────────────────────────────────
def js_update(pct, text, eta_text=""):
    display(Javascript(f'''
        document.getElementById("progress-bar").style.width = "{pct}%";
        document.getElementById("progress-text").innerText = "{text}";
        document.getElementById("eta-text").innerText = "{eta_text}";
    '''), display_id=True)

def js_alert(msg):
    display(Javascript(f'alert("{msg}");'), display_id=True)
    print(msg)

# ─── Main Pipeline ────────────────────────────────────
def run_pipeline(srt_content, voice_key):
    lines = parse_srt(srt_content)
    total = len(lines)

    if total == 0:
        return js_alert('Khong parse duoc dong SRT nao. Kiem tra lai dinh dang.')
    if total > 500:
        return js_alert(f'{total} dong — qua nhieu! Gioi han 500 dong/lan.')
    if voice_key not in VOICE_PROMPTS:
        return js_alert(f'Giong "{voice_key}" khong kha dung.')

    vp = VOICE_PROMPTS[voice_key]
    vn = VOICE_NAMES.get(voice_key, voice_key)
    tmpdir = tempfile.mkdtemp(prefix='srt_')
    timed_files = []
    t0 = time.time()

    js_update(0, f'{vn}: 0/{total}', 'Dang khoi dong...')

    for i, line in enumerate(lines):
        tts_path   = os.path.join(tmpdir, f'tts_{i:04d}.wav')
        timed_path = os.path.join(tmpdir, f'timed_{i:04d}.wav')

        # 1. TTS
        try:
            audio = model.generate(
                text=line['text'], voice_clone_prompt=vp,
                language='vi', speed=0.95, generation_config=GEN_CFG
            )[0]
            write_wav(tts_path, audio, SAMPLING_RATE)
        except Exception as e:
            print(f'  TTS fail dong {i+1}: {e}')
            continue

        # 2. Trim + adelay
        dur_s = max(line['end_s'] - line['start_s'], 0.1)
        delay_ms = int(line['start_s'] * 1000)

        r = subprocess.run([
            'ffmpeg', '-y', '-v', 'error',
            '-i', tts_path,
            '-t', str(dur_s),
            '-af', (f'aresample={SAMPLING_RATE},'
                    f'aformat=sample_fmts=s16:channel_layouts=mono,'
                    f'adelay={delay_ms}:all=1'),
            '-c:a', 'pcm_s16le', timed_path
        ])
        if r.returncode != 0:
            os.remove(tts_path); continue

        timed_files.append(timed_path)
        os.remove(tts_path)

        # 3. Progress (every N lines)
        if (i+1) % max(1, total//50) == 0 or i == total-1:
            pct = int((i+1)/total*100)
            e = time.time()-t0
            eta = (e/(i+1))*(total-i-1) if i>0 else 0
            js_update(pct, f'{i+1}/{total} dong ({pct}%) — {vn}',
                      f'Da chay: {int(e//60)}p{int(e%60)}s · Con lai: ~{int(eta//60)}p{int(eta%60)}s')

    if not timed_files:
        return js_alert('Pipeline that bai — khong co segment nao duoc tao.')

    # 4. Pairwise amix (2 inputs/lan, tranh OOM)
    js_update(100, f'Dang mix {len(timed_files)} segments...', '')
    current = timed_files[0]
    for j, nf in enumerate(timed_files[1:], 1):
        mixed = os.path.join(tmpdir, f'_mix_{j:04d}.wav')
        r = subprocess.run([
            'ffmpeg', '-y', '-v', 'error',
            '-i', current, '-i', nf,
            '-filter_complex', '[0:a][1:a]amix=inputs=2:duration=longest:normalize=0[a]',
            '-map', '[a]', '-c:a', 'pcm_s16le', mixed
        ])
        if r.returncode == 0:
            if os.path.exists(current): os.remove(current)
            if current != nf: os.remove(nf)
            current = mixed

    # 5. Convert MP3
    output_mp3 = os.path.join(tmpdir, 'output.mp3')
    subprocess.run([
        'ffmpeg', '-y', '-v', 'error',
        '-i', current, '-codec:a', 'libmp3lame', '-b:a', '128k', output_mp3
    ])

    # 6. Verify
    probe = subprocess.run([
        'ffprobe', '-v', 'error', '-show_entries', 'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1', output_mp3
    ], capture_output=True, text=True)
    adur = float(probe.stdout.strip()) if probe.returncode == 0 else 0
    smb  = os.path.getsize(output_mp3)/(1024*1024)
    tel  = time.time()-t0

    # 7. Save copy to fixed path (CELL 3 can access)
    os.makedirs("/tmp/srt_output", exist_ok=True)
    subprocess.run(['cp', output_mp3, '/tmp/srt_output/output.mp3'])

    js_update(100, f'Hoan thanh! {adur:.0f}s · {smb:.1f}MB',
              f'Tong: {int(tel//60)}p{int(tel%60)}s · {total} dong · {vn}')

    files.download(output_mp3)

    # Cleanup temp dir (keep /tmp/srt_output/output.mp3)
    import shutil
    shutil.rmtree(tmpdir, ignore_errors=True)
    print(f'DONE — {total} dong · {smb:.1f}MB · {int(tel//60)}p{int(tel%60)}s')
    print(f'💡 Muon chuan hoa am luong EBU R128? Chay CELL 3 ben duoi.')

# ─── Python Callback (goi tu JavaScript) ──────────────
def on_start(voice_key, srt_text):
    display(Javascript('''
        document.getElementById("progress-section").style.display = "block";
        document.getElementById("start-btn").disabled = true;
        document.getElementById("start-btn").innerText = "Dang xu ly...";
    '''), display_id=True)
    run_pipeline(srt_text, voice_key)
    display(Javascript('''
        document.getElementById("start-btn").disabled = false;
        document.getElementById("start-btn").innerText = "Bat dau tao giong doc";
    '''), display_id=True)

output.register_callback('on_start', on_start)

# ─── HTML UI ──────────────────────────────────────────
voice_options = ''.join(
    f'<option value="{k}"{" selected" if k=="nam-tram-am" else ""}>{v}</option>'
    for k,v in VOICE_NAMES.items()
)

display(HTML(f'''
<div style="font-family:Manrope,sans-serif;max-width:640px;margin:30px auto;
            background:#1a1a2e;border-radius:16px;padding:28px;color:#e0e0e0">
  <h2 style="color:#5B3DF6;margin:0 0 4px;font-size:20px">SRT Studio — KIENDOANTTS</h2>
  <p style="margin:0 0 20px;color:#a0a0b0;font-size:14px">
     Chon giong · Dan SRT · Tao giong doc hang loat — giu nguyen timeline
  </p>

  <label style="font-size:13px;color:#a0a0b0;margin-bottom:6px;display:block">
    Chon giong doc
  </label>
  <select id="voice-select"
    style="width:100%;padding:12px;border-radius:12px;margin-bottom:16px;
           background:#0f0f23;color:#e0e0e0;border:1px solid #2a2a4a;
           font:14px Manrope,sans-serif">
    {voice_options}
  </select>

  <label style="font-size:13px;color:#a0a0b0;margin-bottom:6px;display:block">
    Noi dung SRT
  </label>
  <textarea id="srt-input"
    placeholder="1&#10;00:00:00,866 --&gt; 00:00:04,333&#10;Chao mung ban den voi video...&#10;&#10;2&#10;00:00:04,500 --&gt; 00:00:07,200&#10;Hom nay toi se gioi thieu..."
    style="width:100%;height:200px;border-radius:12px;padding:14px;
           font:13px 'JetBrains Mono',monospace;background:#0f0f23;
           color:#e0e0e0;border:1px solid #2a2a4a;resize:vertical;
           box-sizing:border-box"></textarea>

  <p style="text-align:center;color:#a0a0b0;margin:12px 0;font-size:13px">— hoac —</p>

  <button id="upload-btn"
    style="width:100%;padding:12px;border-radius:12px;border:1.5px dashed #5B3DF6;
           background:transparent;color:#5B3DF6;font:14px Manrope,sans-serif;
           cursor:pointer">
    Tai file .srt len
  </button>

  <button id="start-btn"
    style="width:100%;margin-top:14px;padding:14px;border-radius:12px;
           border:none;background:#5B3DF6;color:#fff;font:15px Manrope,sans-serif;
           font-weight:600;cursor:pointer">
    Bat dau tao giong doc
  </button>

  <div id="progress-section" style="display:none;margin-top:20px">
    <div style="background:#2a2a4a;border-radius:8px;height:10px;overflow:hidden">
      <div id="progress-bar"
        style="background:linear-gradient(90deg,#5B3DF6,#8B6FFF);width:0%;
               height:100%;border-radius:8px;transition:width .3s"></div>
    </div>
    <p id="progress-text" style="text-align:center;margin-top:10px;font-size:14px;color:#c0c0d0"></p>
    <p id="eta-text" style="text-align:center;font-size:13px;color:#8080a0"></p>
  </div>
</div>

<script>
// Upload button - pure JS file reading
document.getElementById('upload-btn').onclick = function() {{
    var input = document.createElement('input');
    input.type = 'file';
    input.accept = '.srt,.txt';
    input.onchange = function(e) {{
        var file = e.target.files[0];
        if (!file) return;
        var reader = new FileReader();
        reader.onload = function(e) {{
            document.getElementById('srt-input').value = e.target.result;
        }};
        reader.readAsText(file);
    }};
    input.click();
}};

// Start button - invoke Python callback
document.getElementById('start-btn').onclick = async function() {{
    var voice = document.getElementById('voice-select').value;
    var srt = document.getElementById('srt-input').value;
    if (!srt.trim()) {{
        alert('Vui long nhap noi dung SRT hoac tai file .srt len truoc.');
        return;
    }}
    await google.colab.kernel.invokeFunction('on_start', [voice, srt], {{}});
}};
</script>
'''))

print('Giao dien da san sang!')
print('Chon giong -> Dan SRT (hoac tai file) -> Bam "Bat dau tao giong doc"')

In [ ]:
# ============================================================
# CELL 1: Cai dat dependencies + Tai 7 voice samples
# ============================================================
print('📦 Dang cai dat dependencies (~1 phut)...')
!pip install -q omnivoice "numpy<2.1" "requests==2.32.4"
!apt-get install -qq ffmpeg > /dev/null 2>&1
print('✅ Dependencies da san sang!')

# ─── Voice metadata ────────────────────────────────────
VOICE_KEYS = [
    "nam-cong-nghe", "minh-anh", "thanh-nien-tu-tin",
    "nam-tram-am", "adam", "ngoc-huyen", "nho-ngot-ngao"
]
VOICE_NAMES = {
    "nam-cong-nghe":     "Nam cong nghe",
    "minh-anh":          "Minh Anh",
    "thanh-nien-tu-tin": "Thanh nien tu tin",
    "nam-tram-am":       "Nam tram am",
    "adam":              "Adam",
    "ngoc-huyen":        "Ngoc Huyen",
    "nho-ngot-ngao":     "Nho Ngot Ngao",
}
BASE_URL = "https://raw.githubusercontent.com/doanquangkien/voice-notebooks/main/samples"

import os
os.makedirs("/tmp/srt_voices", exist_ok=True)

# ─── Tai 7 voice samples (~100KB/file) ──────────────────
print('\n🎤 Dang tai voice samples...')
for key in VOICE_KEYS:
    url  = f"{BASE_URL}/{key}.mp3"
    path = f"/tmp/srt_voices/{key}.mp3"
    if not os.path.exists(path):
        !wget -q {url} -O {path}
        print(f'  ✅ {VOICE_NAMES[key]}')
print('✅ Tat ca 7 voice samples da san sang!')
print('⏩ Chay cell tiep theo de khoi dong model (~5 phut lan dau)...')

# SRT Studio — KIENDOANTTS

> **GPU:** T4 16GB · **Lần đầu:** ~5 phút · **Lần sau:** ~2 phút
> Chuyển phụ đề SRT thành giọng đọc AI — giữ nguyên timeline gốc

## Hướng dẫn

1. **Runtime** → **Run all** (Ctrl+F9) — đợi model load (~5 phút lần đầu)
2. Chọn giọng đọc → Dán nội dung SRT (hoặc tải file .srt)
3. Click **🚀 Bắt đầu tạo giọng đọc**
4. File MP3 tự động tải về khi hoàn thành

> ⚠️ Giới hạn: ~500 dòng/lần (Colab T4 16GB RAM). Không upload nội dung nhạy cảm.
> 🔧 Cần thêm giọng? fork repo `srt-notebooks` → thêm sample MP3 → PR.